# Scanning proteins for motif occurrences with FIMO (`scan_motif`)

`scan_motif` searches candidate protein sequences for statistically significant occurrences of a sequence motif described by a Position Weight Matrix (PWM). It is a thin wrapper around FIMO (Find Individual Motif Occurrences) from the MEME (Multiple Em for Motif Elicitation) Suite, run through its Command-Line Interface (CLI), so the MEME Suite (the `fimo` binary) must be installed separately (e.g. `conda install -c bioconda meme`) in addition to `aaanalysis[pro]`.

You bring a PWM (what the motif looks like) and a set of proteins, and `scan_motif` asks FIMO where the motif occurs and how surprising each occurrence is under a background amino-acid model. It keeps the windows whose match p-value is below your threshold, ranks them from most to least significant, and returns them as ready-to-label training windows.

Compared with the pure-Python `AAWindowSampler.sample_motif_matched`, which selects windows by a raw per-position PWM sum and needs no external tool, `scan_motif` uses FIMO's probabilistic, background-aware scoring (a p-value). The two therefore select different windows: reach for `scan_motif` when you want significance-calibrated hits, and for the sampler when you want a dependency-free PWM-sum scan.

It returns one row per matched window (default `output_mode='segments'`): the standard window schema plus FIMO's `motif_score` (a log-odds score) and a `p_value` column.

In [1]:
import pandas as pd
import aaanalysis as aa
import aaanalysis.utils as ut

aa.options["verbose"] = False

# A tiny example with three proteins. 'P1' is a known positive (its 'pos' entry
# lists residue 3), so it is EXCLUDED from the scan; 'C1' and 'C2' are candidates.
df_seq = pd.DataFrame({
    "entry": ["P1", "C1", "C2"],
    "sequence": ["MKLVAAAAAGTR", "QWEAAAAACDEF", "GHIKAAAAALMN"],
    "pos": [[3], [], []],
})

# A Position Weight Matrix (5 positions x 20 amino acids) favouring an
# Alanine ('A') run. Columns are the 20 canonical amino acids.
motif_pwm = pd.DataFrame(0.01, index=range(5), columns=list(ut.LIST_CANONICAL_AA))
motif_pwm["A"] = 0.81

# Smallest call: scan the candidates for significant matches of the motif.
df_hits = aa.scan_motif(df_seq=df_seq, motif_pwm=motif_pwm, window_size=5,
                        pvalue_threshold=1e-2)
aa.display_df(df_hits, show_shape=True)

DataFrame shape: (12, 10)


,entry_win,entry,sequence,window,source_position,label,role,strategy,motif_score,p_value
1,C1_4-8,C1,QWEAAAAACDEF,AAAAA,6,0,Negative,motif_matched,16.719700,0.000002
2,C2_5-9,C2,GHIKAAAAALMN,AAAAA,7,0,Negative,motif_matched,16.719700,0.000002
3,C1_5-9,C1,QWEAAAAACDEF,AAAAC,7,0,Negative,motif_matched,12.617800,0.000007
4,C2_4-8,C2,GHIKAAAAALMN,KAAAA,6,0,Negative,motif_matched,11.356700,0.000075
5,C1_3-7,C1,QWEAAAAACDEF,EAAAA,5,0,Negative,motif_matched,11.299400,0.000092
6,C2_6-10,C2,GHIKAAAAALMN,AAAAL,8,0,Negative,motif_matched,10.980900,0.000135
7,C1_2-6,C1,QWEAAAAACDEF,WEAAA,4,0,Negative,motif_matched,7.579620,0.000238
8,C1_6-10,C1,QWEAAAAACDEF,AAACD,8,0,Negative,motif_matched,7.369430,0.000286
9,C2_7-11,C2,GHIKAAAAALMN,AAALM,9,0,Negative,motif_matched,6.592360,0.000787
10,C2_3-7,C2,GHIKAAAAALMN,IKAAA,5,0,Negative,motif_matched,6.025480,0.001960


## Selecting candidates: `df_seq` and `pos_col`

`df_seq` holds the proteins (`entry` + `sequence` columns). `pos_col` (default `'pos'`) decides which rows are scanned: rows with a **non-empty** position are treated as **known positives and excluded**, while rows with an empty cell are the **candidates** FIMO scans. Note that `P1` contains the same `AAAAA` motif as the candidates, yet never appears in the results — because it is a positive.

In [2]:
print("Candidate entries with a hit:", sorted(df_hits["entry"].unique()))
print("'P1' excluded despite containing the motif:", "P1" not in set(df_hits["entry"]))

Candidate entries with a hit: ['C1', 'C2']
'P1' excluded despite containing the motif: True


## The motif and its significance: `motif_pwm`, `window_size`, `pvalue_threshold`

`motif_pwm` is the motif (shape `(window_size, 20)`) and `window_size` must equal its number of rows. `pvalue_threshold` is FIMO's significance cutoff (a probability in `[0, 1]`, mapped to `fimo --thresh`): **smaller is stricter**, keeping fewer but more significant windows.

In [3]:
strict = aa.scan_motif(df_seq=df_seq, motif_pwm=motif_pwm, window_size=5,
                       pvalue_threshold=1e-4)
loose = aa.scan_motif(df_seq=df_seq, motif_pwm=motif_pwm, window_size=5,
                      pvalue_threshold=1e-1)
print(f"strict (p<1e-4): {len(strict)} hits;  loose (p<1e-1): {len(loose)} hits")

strict (p<1e-4): 5 hits;  loose (p<1e-1): 15 hits


## Output shape and labels: `output_mode`, `n`, `role`, `label_test`, `label_ref`

`output_mode` picks the schema: `'segments'` (default) returns one row per window; `'sequences'` returns one row per protein with a per-residue `labels` list. `n` caps how many windows are returned (most significant first). `role`, `label_ref`, and `label_test` tag the rows for positive-unlabeled (PU) learning / hard-negative-mining workflows: `label_ref`/`role` mark the scanned windows, and (in `'sequences'` mode) `label_test` marks the known-positive positions.

In [4]:
# Segments mode: cap to the 3 most significant windows, tag them as negatives.
df_seg = aa.scan_motif(df_seq=df_seq, motif_pwm=motif_pwm, window_size=5,
                       pvalue_threshold=1e-2, output_mode="segments",
                       n=3, role="Negative", label_ref=0)
aa.display_df(df_seg, show_shape=True)

# Sequences mode: per-residue labels (positives = label_test, hits = label_ref).
df_lab = aa.scan_motif(df_seq=df_seq, motif_pwm=motif_pwm, window_size=5,
                       pvalue_threshold=1e-2, output_mode="sequences",
                       label_test=1, label_ref=0)
aa.display_df(df_lab, show_shape=True)

DataFrame shape: (3, 10)


,entry_win,entry,sequence,window,source_position,label,role,strategy,motif_score,p_value
1,C1_4-8,C1,QWEAAAAACDEF,AAAAA,6,0,Negative,motif_matched,16.719700,0.000002
2,C2_5-9,C2,GHIKAAAAALMN,AAAAA,7,0,Negative,motif_matched,16.719700,0.000002
3,C1_5-9,C1,QWEAAAAACDEF,AAAAC,7,0,Negative,motif_matched,12.617800,0.000007


DataFrame shape: (3, 3)


,entry,sequence,labels
1,P1,MKLVAAAAAGTR,"[None, None, 1, None, None, None, None, None, None, None, None, None]"
2,C1,QWEAAAAACDEF,"[None, None, 0, 0, 0, 0, 0, 0, 0, None, None, None]"
3,C2,GHIKAAAAALMN,"[None, None, None, None, 0, 0, 0, 0, 0, None, None, None]"


## Tuning FIMO's scoring: `motif_pseudo`, `max_stored_scores`, `bg_file`

These pass straight through to FIMO and change how it scores occurrences:

* `motif_pseudo` — pseudocount added to the motif before scoring (FIMO default `0.1`); pass `0.0` to disable smoothing.
* `max_stored_scores` — cap on the number of occurrences FIMO stores internally (FIMO default `100000`); raise it only for very large candidate sets.
* `bg_file` — path to a MEME-format background amino-acid frequency file; when omitted, FIMO uses its built-in protein background. Changing the background changes the match p-values.

In [5]:
df_tuned = aa.scan_motif(df_seq=df_seq, motif_pwm=motif_pwm, window_size=5,
                         pvalue_threshold=1e-2, motif_pseudo=0.0,
                         max_stored_scores=1_000_000)
aa.display_df(df_tuned, show_shape=True)

DataFrame shape: (13, 10)


,entry_win,entry,sequence,window,source_position,label,role,strategy,motif_score,p_value
1,C1_4-8,C1,QWEAAAAACDEF,AAAAA,6,0,Negative,motif_matched,17.330800,0.000002
2,C2_5-9,C2,GHIKAAAAALMN,AAAAA,7,0,Negative,motif_matched,17.330800,0.000002
3,C1_5-9,C1,QWEAAAAACDEF,AAAAC,7,0,Negative,motif_matched,13.000000,0.000007
4,C2_4-8,C2,GHIKAAAAALMN,KAAAA,6,0,Negative,motif_matched,11.315800,0.000075
5,C1_3-7,C1,QWEAAAAACDEF,EAAAA,5,0,Negative,motif_matched,11.225600,0.000092
6,C2_6-10,C2,GHIKAAAAALMN,AAAAL,8,0,Negative,motif_matched,10.669200,0.000135
7,C1_2-6,C1,QWEAAAAACDEF,WEAAA,4,0,Negative,motif_matched,7.345860,0.000243
8,C1_6-10,C1,QWEAAAAACDEF,AAACD,8,0,Negative,motif_matched,7.165410,0.000307
9,C2_7-11,C2,GHIKAAAAALMN,AAALM,9,0,Negative,motif_matched,6.000000,0.000960
10,C2_3-7,C2,GHIKAAAAALMN,IKAAA,5,0,Negative,motif_matched,5.353380,0.001960
